In [1]:
import pandas as pd

from churnlens.features.preprocessing import FEATURES, TARGET

train = pd.read_csv("../data/processed/train.csv")
X_train = train[FEATURES]
y_train = train[TARGET]

spw = (y_train == 0).sum() / (y_train == 1).sum()
print(X_train.shape, "churn rate:", round(y_train.mean(), 3), "scale_pos_weight:", round(spw, 2))

(5634, 19) churn rate: 0.265 scale_pos_weight: 2.77


In [2]:
from sklearn.model_selection import StratifiedKFold, cross_validate

from churnlens.models.pipelines import MODEL_NAMES, build_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

rows = []
for name in MODEL_NAMES:
    pipe = build_pipeline(name, scale_pos_weight=spw)
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    row = {"model": name}
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        row[metric] = f"{vals.mean():.3f} +/- {vals.std():.3f}"
    rows.append(row)

results = pd.DataFrame(rows).set_index("model")
results

d:\Gauriii\Projects\ChurnLens\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Gauriii\Projects\ChurnLens\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Gauriii\Projects\ChurnLens\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Gauriii\Projects

,roc_auc,pr_auc,precision,recall,f1
model,,,,,
dummy,0.500 +/- 0.000,0.265 +/- 0.000,0.000 +/- 0.000,0.000 +/- 0.000,0.000 +/- 0.000
logistic_regression,0.846 +/- 0.012,0.660 +/- 0.019,0.517 +/- 0.019,0.801 +/- 0.038,0.628 +/- 0.023
random_forest,0.845 +/- 0.009,0.655 +/- 0.020,0.539 +/- 0.012,0.766 +/- 0.028,0.632 +/- 0.014
xgboost,0.846 +/- 0.011,0.667 +/- 0.018,0.525 +/- 0.015,0.793 +/- 0.027,0.632 +/- 0.019
